In [1]:
# need to set up GEE

import ee

print(ee.__version__)
ee.Authenticate()
ee.Initialize()




1.7.1


In [49]:
# set up functions



def sample_global_land_points(
    n_points: int,
    seed: int | None = None,
    scale: int = 1000,
) -> ee.FeatureCollection:
    """
    Sample roughly uniform random points over global land using ESA WorldCover v200.

    n_points : desired number of points.
    seed     : random seed for reproducibility.
    scale    : sampling scale in meters.
    """
    # ESA WorldCover v200
    worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()
    land_mask = worldcover.select("Map").neq(80)  # 80 = permanent water

    # Global rectangle in WGS84
    global_region = ee.Geometry.Rectangle(
        [-180, -90, 180, 90],
        proj="EPSG:4326",
        geodesic=False,
    )

    # Random image in [0,1), masked to land only
    random_img = ee.Image.random(seed=seed).updateMask(land_mask).rename("rand")

    # --- Key change: oversample, then randomly keep n_points ---

    oversample = n_points * 10  # sample more than needed

    samples = random_img.sample(
        region=global_region,
        scale=scale,
        numPixels=oversample,
        geometries=True,
        seed=seed,
    )

    # Randomize the samples and take exactly n_points
    samples = samples.randomColumn(columnName="r", seed=seed).sort("r").limit(n_points)

    return samples

In [50]:


def generate_snic_tile(
    lat: float,
    lon: float,
    year: int,
    half_width_km: float = 3.0,
    snic_size: int = 10,
) -> tuple[ee.Image, ee.Geometry]:
    """
    Generate a SNIC clusters image around a given lat/lon using Landsat 8
    Collection 2, Level-2 SR.

    Returns:
        clusters (ee.Image), study_area (ee.Geometry)
    """
    # 1. Study area
    point = ee.Geometry.Point([lon, lat])
    study_area = point.buffer(half_width_km * 1000).bounds()

    # 2. Date range (mid-summer window)
    start_date = f"{year}-07-01"
    end_date   = f"{year}-08-31"

    # 3. Landsat 8 C2 L2 collection
    l8_c2 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")

    # 4. Cloud/shadow mask for Collection 2 (QA_PIXEL)
    def mask_l8_c2_sr(image):
        qa = image.select("QA_PIXEL")
        cloud_bit        = 1 << 3   # cloud
        cloud_shadow_bit = 1 << 4   # cloud shadow

        mask = (qa.bitwiseAnd(cloud_bit).eq(0)
                  .And(qa.bitwiseAnd(cloud_shadow_bit).eq(0)))

        return (image
                .updateMask(mask)
                .select(["SR_B4", "SR_B5", "SR_B7"]))  # RED, NIR, SWIR2

    # 5. Composite
    composite = (l8_c2
        .filterBounds(study_area)
        .filterDate(start_date, end_date)
        .map(mask_l8_c2_sr)
        .median()
        .clip(study_area))

    # 6. SNIC segmentation
    snic = ee.Algorithms.Image.Segmentation.SNIC(
        image=composite,
        size=snic_size
    )

    clusters = snic.select(["clusters"])
    return clusters, study_area
    #return composite, study_area

In [56]:
# the downloads end up as zip files that we need to then extract
import zipfile

import os

def extract_and_rename_geotiff(zip_path, delete_zip=True):
    """
    Extracts a GeoTIFF from an Earth Engine ZIP download and renames it
    to match the ZIP filename (e.g. foo.zip → foo.tif).

    Parameters
    ----------
    zip_path : str
        Path to the .zip file downloaded from Earth Engine.
    delete_zip : bool
        If True, remove the zip file after extraction.

    Returns
    -------
    out_tif_path : str
        Full path to the renamed GeoTIFF.
    """

    zip_path = os.path.abspath(zip_path)
    out_dir = os.path.dirname(zip_path)
    base = os.path.splitext(os.path.basename(zip_path))[0]
    target_tif = os.path.join(out_dir, base + ".tif")

    if not os.path.exists(zip_path):
        raise FileNotFoundError(zip_path)

    with zipfile.ZipFile(zip_path, "r") as z:
        # find any .tif inside the zip
        tif_files = [f for f in z.namelist() if f.lower().endswith(".tif")]

        if not tif_files:
            raise RuntimeError(f"No GeoTIFF found inside {zip_path}")

        # take the first TIFF found (Earth Engine usually only has one)
        internal_name = tif_files[0]

        # extract to output directory
        z.extract(internal_name, out_dir)

        extracted_path = os.path.join(out_dir, internal_name)

    # Rename to match ZIP name
    if os.path.exists(target_tif):
        os.remove(target_tif)   # avoid collisions

    os.rename(extracted_path, target_tif)

    # Cleanup leftover internal directory structure if present
    internal_dir = os.path.dirname(extracted_path)
    if internal_dir and internal_dir != out_dir:
        try:
            os.rmdir(internal_dir)  # will only remove if empty
        except OSError:
            pass

    # Optionally delete zip
    if delete_zip:
        os.remove(zip_path)

    print(f"✅ Extracted and renamed: {target_tif}")
    return target_tif


In [51]:
# 
def download_snic_tile_for_point(lat, lon, year, out_path, snic_size=10, half_width_km=3):
   
    
    clusters, study_area = generate_snic_tile(
        lat=lat,
        lon=lon,
        year=year,
        half_width_km=half_width_km,
        snic_size=snic_size
    )

    # Convert the ee.Geometry to a client-side GeoJSON-like dict
    study_area_geojson = study_area.getInfo()  # this is a dict
    # getDownloadURL expects region as a GeoJSON geometry or its 'coordinates'
    # safest is to pass the full geometry dict
    params = {
        "scale": 30,
        "crs": "EPSG:4326",  # or your projected CRS
        "region": study_area_geojson,   # <-- fixed
        "filePerBand": False
    }

    url = clusters.getDownloadURL(params)
    r = requests.get(url, stream=True)
    with open(out_path, "wb") as f:
        for chunk in r.iter_content(1024 * 1024):
            if chunk:
                f.write(chunk)

In [64]:

def retry_ee(fn, max_retries=25, delay=30, backoff=2, verbose=True):
    """
    Retry wrapper for flaky Earth Engine computations.

    Parameters
    ----------
    fn : callable
        A function with NO arguments that runs an EE operation.
    max_retries : int
        Maximum number of attempts.
    delay : seconds
        Initial wait before retry.
    backoff : multiplier
        Delay grows each failure: delay *= backoff
    """

    for attempt in range(1, max_retries + 1):
        try:
            return fn()

        except ee.EEException as e:
            if attempt == max_retries:
                raise

            wait = delay * (backoff ** (attempt - 1))
            if verbose:
                print(f"⚠️ EEException (attempt {attempt}/{max_retries}): {e}")
                print(f"   retrying in {wait:.1f} seconds...")

            time.sleep(wait)

In [65]:
import requests
import os
import time

# Sample 5 random land points
seed = int(time.time())
points = sample_global_land_points(1000, seed=seed, scale=1000)

features = points.toList(points.size())
year = 2019
snic_size = 10

for i in range(points.size().getInfo()):
    feat = ee.Feature(features.get(i))
    lon, lat = feat.geometry().coordinates().getInfo()
    out_path = f"downloaded_images/snic_{snic_size}_{year}_{lat:.3f}_{lon:.3f}.zip"
    print("Generating", out_path)
    retry_ee(lambda:download_snic_tile_for_point(lat, lon, year, out_path, snic_size=snic_size, half_width_km=3))
    extract_and_rename_geotiff(out_path)
    

print("Done")


Generating downloaded_images/snic_10_2019_57.495_63.524.zip
✅ Extracted and renamed: /Users/rkennedy/Dropbox/caol/code/_proj/dist_uncertainty/downloaded_images/snic_10_2019_57.495_63.524.tif
Generating downloaded_images/snic_10_2019_32.147_-83.117.zip
✅ Extracted and renamed: /Users/rkennedy/Dropbox/caol/code/_proj/dist_uncertainty/downloaded_images/snic_10_2019_32.147_-83.117.tif
Generating downloaded_images/snic_10_2019_29.691_4.710.zip
✅ Extracted and renamed: /Users/rkennedy/Dropbox/caol/code/_proj/dist_uncertainty/downloaded_images/snic_10_2019_29.691_4.710.tif
Generating downloaded_images/snic_10_2019_28.719_82.915.zip
✅ Extracted and renamed: /Users/rkennedy/Dropbox/caol/code/_proj/dist_uncertainty/downloaded_images/snic_10_2019_28.719_82.915.tif
Generating downloaded_images/snic_10_2019_14.883_44.809.zip
✅ Extracted and renamed: /Users/rkennedy/Dropbox/caol/code/_proj/dist_uncertainty/downloaded_images/snic_10_2019_14.883_44.809.tif
Generating downloaded_images/snic_10_2019_0.1

RefreshError: ('invalid_grant: Token has been expired or revoked.', {'error': 'invalid_grant', 'error_description': 'Token has been expired or revoked.'})

In [24]:
# KEEEP FOR REFERENCE
#  The downloaded files are not working right.  
# Suggested: inspect them using this to see what they are.  
# A super useful function. 

def inspect_file_header(path, nbytes=64):
    print(f"Inspecting {path!r}")
    if not os.path.exists(path):
        print("  File does not exist.")
        return

    with open(path, "rb") as f:
        head = f.read(nbytes)

    print("  First bytes:", head[:32])
    hstrip = head.lstrip()

    if hstrip.startswith(b"II") or hstrip.startswith(b"MM"):
        print("  → Looks like a TIFF/GeoTIFF header.")
    elif hstrip.startswith(b"PK"):
        print("  → Looks like a ZIP file (PK header).")
    elif hstrip.startswith(b"<"):
        print("  → Looks like HTML/XML (probably an error page from EE).")
    elif hstrip.startswith(b"{"):
        print("  → Looks like JSON (also likely an error from EE).")
    else:
        print("  → Unknown header; not obviously a TIFF/ZIP/HTML/JSON.")

# Example:
inspect_file_header("downloaded_images/snic_2019_6.567_-59.446.tif")

Inspecting 'downloaded_images/snic_2019_6.567_-59.446.tif'
  First bytes: b'PK\x03\x04\n\x00\x00\x08\x00\x00\x0cf\x84[b3\x81\x8a\x17\x1f\x00\x00\x17\x1f\x00\x00\x17\x00\x00\x00SR'
  → Looks like a ZIP file (PK header).
